# SACR Tool — Complete Sentiment Analysis Pipeline (Fixed)

A comprehensive notebook covering all 4 phases:
1. **Data Preprocessing & EDA** — Load, clean, explore, visualize
2. **Feature Engineering** — Text vectorization, label creation
3. **Model Selection** — Train multiple classifiers
4. **Model Evaluation & XAI** — Confusion matrix, ROC-AUC, misclassification analysis, feature importance

### What changed vs. the original pipeline
- **Neutral sentiment is no longer silently dropped.** A config flag (`INCLUDE_NEUTRAL`) lets you keep a 3-class negative/neutral/positive setup for rating-scale datasets, instead of always collapsing to binary.
- **Fixed the data-leakage bug that produced accuracy = 1.0.** The vectorizer (TF-IDF/CountVectorizer) was previously fit on the *entire* dataset (`X = vect.fit_transform(all_data)`) before the train/test split — so the vocabulary, IDF weights, and `min_df` cutoffs were computed using test-set text. The split now happens on raw text **first**, and the vectorizer is `fit` only on the training split, then `transform`-ed on the test split.
- **Works across binary, multi-class, balanced and imbalanced datasets.** Labels are detected generically (categorical string labels, binary 0/1, multi-point rating scales, or continuous scores), encoded with `LabelEncoder`, and every downstream metric/plot (classification report, confusion matrix, ROC-AUC, feature importance, error analysis) is written to work for however many classes are actually present — not hardcoded to 2.
- **Imbalance-aware by default:** stratified train/test split, `class_weight='balanced'` on the classifiers that support it, and both macro- and weighted-averaged F1 reported (macro F1 will expose a model that's just predicting the majority class, which weighted F1/accuracy can hide).
- **A built-in leakage sanity check** flags any model that scores suspiciously close to perfect accuracy so this doesn't slip by silently again.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, roc_auc_score)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import MultinomialNB

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')


---
## Configuration

Set these once per dataset. Everything downstream adapts automatically.

In [ ]:
# --- Pipeline Configuration ---
import ipywidgets as widgets
from IPython.display import display
import io as _io

uploader = widgets.FileUpload(accept=".csv,.xlsx,.json,.txt", multiple=False)
display(uploader)

if uploader.value:
    uploaded = list(uploader.value.values())[0]
    content = uploaded["content"]
    name = uploaded["metadata"]["name"]
    if name.endswith(".csv"):
        df = pd.read_csv(_io.BytesIO(content))
    elif name.endswith(".xlsx"):
        df = pd.read_excel(_io.BytesIO(content))
    elif name.endswith(".json"):
        df = pd.read_json(_io.BytesIO(content))
    else:
        df = pd.read_csv(_io.BytesIO(content), sep="\t")
    print(f"Loaded uploaded file: {name}")
else:
    DATA_PATH = r"D:\\research\\dataset\\IMDB-Dataset.csv"
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded default file: {DATA_PATH}")

INCLUDE_NEUTRAL = True
TEST_SIZE = 0.3
RANDOM_STATE = 42


---
## Phase 1: Data Preprocessing & EDA

In [ ]:
# --- Load Dataset (data loaded in config cell above) ---
print(f'Shape: {df.shape}')
df.head(10)


In [ ]:
# --- Basic Info & Missing Values ---
print('=== Info ===')
df.info()
print(f'\n=== Missing Values ===\n{df.isnull().sum()}')


In [ ]:
# --- Detect text & target columns automatically ---
text_col = None
for col in df.columns:
    if df[col].dtype == 'object':
        avg_len = df[col].astype(str).str.len().mean()
        if avg_len > 50:
            text_col = col
            break

potential_sentiment_cols = [col for col in df.columns
                             if ('sentiment' in col.lower() or 'label' in col.lower())
                             and df[col].dtype == 'object']
rating_cols = [col for col in df.columns if 'rating' in col.lower() or 'score' in col.lower()]

print(f'Text column detected: {text_col}')
print(f'Potential categorical sentiment columns: {potential_sentiment_cols}')
print(f'Numeric rating/score columns: {rating_cols}')


In [ ]:
# --- Drop rows with null text ---
before = len(df)
df = df.dropna(subset=[text_col])
print(f'Dropped {before - len(df)} rows with null text')


### Label creation (generalized)

This replaces the old logic that always mapped ratings to binary positive/negative and
silently threw away the neutral band. Now:
- A categorical column (e.g. `sentiment` with values like `positive`/`negative`/`neutral`) is used as-is, preserving however many classes it has.
- An existing 0/1 column is mapped to negative/positive.
- A rating scale (e.g. 1-10) is mapped to negative / neutral / positive, and the neutral band is **kept** unless `INCLUDE_NEUTRAL = False`.
- A continuous score with no obvious scale is binarized at the median as a last resort.
- Keyword-based labeling is only used if no target column can be found at all (rare, and weaker than a real label column).

In [ ]:
# --- Label Creation ---
target_col = None
label_source = None

if potential_sentiment_cols:
    target_col = potential_sentiment_cols[0]
    label_source = 'categorical'
elif rating_cols:
    target_col = rating_cols[0]
    label_source = 'numeric'

if target_col is None:
    df['label_raw'] = np.where(
        df[text_col].astype(str).str.contains(r'\b(good|excellent|positive|amazing|great|wonderful)\b',
                                                case=False, na=False),
        'positive', 'negative'
    )
    print('No target column found at all - used keyword-based labeling as a last resort. '
          'This is a weak fallback; a real label column is strongly preferred.')

elif label_source == 'categorical':
    df['label_raw'] = df[target_col].astype(str).str.strip().str.lower()
    print(f'Using categorical target column "{target_col}" directly. '
          f'Values found: {sorted(df["label_raw"].unique())}')

else:
    vals = df[target_col].dropna()
    unique_vals = sorted(vals.unique())
    if set(unique_vals) <= {0, 1}:
        df['label_raw'] = df[target_col].map({0: 'negative', 1: 'positive'})
        print(f'Target column "{target_col}" is already binary (0/1) - mapped to negative/positive.')
    elif vals.max() > 1:
        def map_rating(x):
            if x >= 7:
                return 'positive'
            elif x <= 4:
                return 'negative'
            else:
                return 'neutral'
        df['label_raw'] = df[target_col].apply(map_rating)
        n_neutral = (df['label_raw'] == 'neutral').sum()
        if INCLUDE_NEUTRAL:
            print(f'Rating scale detected in "{target_col}". Mapped >=7->positive, <=4->negative, '
                  f'5-6->neutral. Keeping {n_neutral} neutral rows as a 3rd class (INCLUDE_NEUTRAL=True).')
        else:
            df = df[df['label_raw'] != 'neutral']
            print(f'Rating scale detected in "{target_col}". Mapped >=7->positive, <=4->negative, '
                  f'dropped {n_neutral} neutral rows (INCLUDE_NEUTRAL=False).')
    else:
        med = vals.median()
        df['label_raw'] = np.where(df[target_col] > med, 'positive', 'negative')
        print(f'Continuous score column "{target_col}" with no obvious scale - binarized at median={med}.')

df = df.dropna(subset=['label_raw']).copy()
print(f'\nLabel distribution:\n{df["label_raw"].value_counts()}')


In [ ]:
# --- Encode labels (works for 2, 3, or N classes) ---
le = LabelEncoder()
df['label'] = le.fit_transform(df['label_raw'])
class_names = list(le.classes_)
n_classes = len(class_names)

print(f'Classes: {class_names}  (n_classes={n_classes})')
print(f'Class balance:\n{df["label"].value_counts(normalize=True).round(3)}')

imbalance_ratio = df['label'].value_counts().max() / df['label'].value_counts().min()
if imbalance_ratio > 1.5:
    print(f'\nNote: classes are imbalanced (majority/minority ratio = {imbalance_ratio:.2f}). '
          f'The split below is stratified and models use class_weight="balanced" where supported.')


In [ ]:
# --- Text Length Distribution ---
df['text_length'] = df[text_col].astype(str).apply(len)
df['word_count'] = df[text_col].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['text_length'], bins=50, edgecolor='black')
axes[0].set_title('Text Length Distribution')
axes[0].set_xlabel('Length (chars)')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=50, edgecolor='black', color='orange')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# --- Class Distribution ---
print(f'\n=== Label Distribution ===')
print(df['label_raw'].value_counts())

fig, ax = plt.subplots(figsize=(6, 4))
df['label_raw'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Class Distribution')
ax.set_xlabel('label')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# --- Word Cloud per class (works for any number of classes) ---
from wordcloud import WordCloud

n_show = min(n_classes, 4)
fig, axes = plt.subplots(1, n_show, figsize=(8 * n_show, 7))
if n_show == 1:
    axes = [axes]

colormaps = ['Greens', 'Reds', 'Blues', 'Oranges']
for i, cname in enumerate(class_names[:n_show]):
    text_sample = ' '.join(df[df['label_raw'] == cname][text_col].astype(str).head(500))
    if not text_sample.strip():
        axes[i].axis('off')
        continue
    wc = WordCloud(width=500, height=400, background_color='white',
                   colormap=colormaps[i % len(colormaps)], max_words=100).generate(text_sample)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].axis('off')
    axes[i].set_title(f'{cname.title()} Reviews')
plt.tight_layout()
plt.show()


---
## Phase 2: Feature Engineering

Text cleaning, stopword customization (keep 'not' for sentiment, remove modals), contraction
expansion, lemmatization. **The train/test split now happens on raw text before vectorization,
and the vectorizer is fit only on the training split** — this is the key fix for the data-leakage
bug that was producing accuracy = 1.0.

In [ ]:
# --- Customized Stopwords ---
# Keep 'not' (critical for sentiment), remove modal verbs that don't carry sentiment
stop_words = stopwords.words('english')
new_stopwords = ['would', 'shall', 'could', 'might']
stop_words.extend(new_stopwords)
stop_words.remove('not')
stop_words = set(stop_words)
print(f'Custom stopwords loaded ({len(stop_words)} words)')


In [ ]:
# --- Contraction Expansion ---
def contraction_expansion(content):
    content = re.sub(r"won\'t", "would not", content)
    content = re.sub(r"can\'t", "can not", content)
    content = re.sub(r"don\'t", "do not", content)
    content = re.sub(r"shouldn\'t", "should not", content)
    content = re.sub(r"needn\'t", "need not", content)
    content = re.sub(r"hasn\'t", "has not", content)
    content = re.sub(r"haven\'t", "have not", content)
    content = re.sub(r"weren\'t", "were not", content)
    content = re.sub(r"mightn\'t", "might not", content)
    content = re.sub(r"didn\'t", "did not", content)
    content = re.sub(r"n\'t", " not", content)
    return content

# --- Text Cleaning Pipeline ---
def remove_special_chars(content):
    return re.sub(r'\W+', ' ', content)

def remove_urls(content):
    return re.sub(r'http\S+', '', content)

def remove_stopwords_from_text(content):
    clean = []
    for i in content.split():
        if i.strip().lower() not in stop_words and i.strip().lower().isalpha():
            clean.append(i.strip().lower())
    return ' '.join(clean)

def data_cleaning(content):
    content = contraction_expansion(content)
    content = remove_special_chars(content)
    content = remove_urls(content)
    content = remove_stopwords_from_text(content)
    return content


In [ ]:
# --- Apply Cleaning ---
print('Cleaning text...')
df['processed_text'] = df[text_col].astype(str).apply(data_cleaning)
df[['processed_text']].head(10)


In [ ]:
# --- Drop rows with empty processed text ---
clean_df = df.dropna(subset=['processed_text', 'label']).copy()
clean_df = clean_df[clean_df['processed_text'].str.strip() != '']
print(f'Clean shape: {clean_df.shape}')


In [ ]:
# --- Train/Test Split on RAW TEXT, done BEFORE vectorization ---
# Splitting before vectorizing (and fitting the vectorizer only on the train rows below)
# is what prevents the vectorizer's vocabulary / IDF weights / min_df cutoffs from ever
# seeing the test set. Stratified on label to keep class proportions consistent, which
# matters most on imbalanced datasets.
train_df, test_df = train_test_split(
    clean_df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=clean_df['label']
)
print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Train class balance:\n{train_df["label_raw"].value_counts(normalize=True).round(3)}')
print(f'Test class balance:\n{test_df["label_raw"].value_counts(normalize=True).round(3)}')


In [ ]:
# --- LemmaTokenizer (lemmatizes during vectorization) ---
class LemmaTokenizer(object):
    def __init__(self):
        self.wordnetlemma = WordNetLemmatizer()
    def __call__(self, reviews):
        return [self.wordnetlemma.lemmatize(word) for word in word_tokenize(reviews)]

# --- Vectorizer Settings ---
VECTORIZER_TYPE = 'TF-IDF'  # or 'CountVectorizer'
NGRAM_MIN, NGRAM_MAX = 1, 3
MIN_DF = 10
MAX_FEATURES = 10000

if VECTORIZER_TYPE == 'CountVectorizer':
    vect = CountVectorizer(analyzer='word', tokenizer=LemmaTokenizer(),
                           ngram_range=(NGRAM_MIN, NGRAM_MAX), min_df=MIN_DF, max_features=MAX_FEATURES)
else:
    vect = TfidfVectorizer(analyzer='word', tokenizer=LemmaTokenizer(),
                            ngram_range=(NGRAM_MIN, NGRAM_MAX), min_df=MIN_DF, max_features=MAX_FEATURES)

# fit_transform on TRAIN ONLY, transform (never fit) on TEST -> no leakage
x_train = vect.fit_transform(train_df['processed_text'])
x_test = vect.transform(test_df['processed_text'])
y_train = train_df['label'].values
y_test = test_df['label'].values

print(f'x_train: {x_train.shape}, x_test: {x_test.shape}')
print(f'y_train distribution: {np.bincount(y_train)}')
print(f'y_test distribution: {np.bincount(y_test)}')


In [ ]:
# --- Top Features by Score (computed from the training vocabulary only) ---
feature_names = vect.get_feature_names_out()
if VECTORIZER_TYPE == 'CountVectorizer':
    feature_scores = np.asarray(x_train.sum(axis=0)).flatten()
    score_type = 'Frequency'
else:
    feature_scores = np.asarray(x_train.mean(axis=0)).flatten()
    score_type = 'Mean TF-IDF'

score_df = pd.DataFrame({'Feature': feature_names, 'Score': feature_scores})\
    .sort_values('Score', ascending=False).head(20)
print(f'Top 20 Features by {score_type} (train set)')
score_df


In [ ]:
# --- Chi-Squared Feature Selection (train set only) ---
from sklearn.feature_selection import chi2
chi_scores, _ = chi2(x_train, y_train)
chi_df = pd.DataFrame({'Feature': feature_names, 'Chi2 Score': chi_scores})\
    .sort_values('Chi2 Score', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x='Chi2 Score', y='Feature', data=chi_df, ax=ax)
ax.set_title('Top 20 Features by Chi-Squared Score')
plt.tight_layout()
plt.show()


---
## Phase 3: Model Selection

Train multiple classifiers and collect metrics. `class_weight='balanced'` is used where the estimator supports it, so imbalanced datasets don't just collapse to predicting the majority class.

In [ ]:
# --- Model Configuration ---
SEED = 42
classifiers = {
    'Logistic Regression': LogisticRegression(C=10, max_iter=200, solver='lbfgs',
                                               random_state=SEED, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=2, criterion='gini',
                                             random_state=SEED, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=2,
                                             random_state=SEED, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=1.0, random_state=SEED),
    'Naive Bayes': MultinomialNB(alpha=1.0)
}

results = []
trained_models = {}


In [ ]:
for name, clf in classifiers.items():
    print(f'\n{"="*50}\nTraining {name}...')
    start = time.time()

    clf.fit(x_train, y_train)
    training_time = time.time() - start

    y_pred = clf.predict(x_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision_w = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall_w = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision_w,
        'Recall': recall_w,
        'F1_Weighted': f1_w,
        'F1_Macro': f1_macro,
        'Training_Time': training_time
    })
    trained_models[name] = clf

    print(f'  Accuracy:      {accuracy:.4f}')
    print(f'  Precision (w): {precision_w:.4f}')
    print(f'  Recall (w):    {recall_w:.4f}')
    print(f'  F1 (weighted): {f1_w:.4f}')
    print(f'  F1 (macro):    {f1_macro:.4f}')
    print(f'  Time:          {training_time:.2f}s')
    print()
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# --- Leakage sanity check ---
print(f'\n{"="*50}')
for r in results:
    if r['Accuracy'] >= 0.999:
        print(f"WARNING: '{r['Model']}' scored {r['Accuracy']:.4f} accuracy - this is suspiciously "
              f"close to perfect and usually signals data leakage (vectorizer fit on test data, "
              f"duplicate rows shared between train/test, or a feature that encodes the label). "
              f"Re-check the pipeline before trusting this result.")


In [ ]:
# --- Results Summary ---
results_df = pd.DataFrame(results).sort_values('F1_Weighted', ascending=False)
results_df


In [ ]:
# --- Performance Comparison Charts ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1_Weighted', 'F1_Macro']
for i, metric in enumerate(metrics):
    sns.barplot(x='Model', y=metric, data=results_df, ax=axes[i])
    axes[i].set_title(f'{metric} Comparison')
    axes[i].tick_params(axis='x', rotation=45)
    for j, v in enumerate(results_df[metric]):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

axes[-1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Training Time Comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='Model', y='Training_Time', data=results_df, ax=ax, palette='viridis')
ax.set_title('Training Time Comparison')
ax.set_ylabel('Time (seconds)')
for i, v in enumerate(results_df['Training_Time']):
    ax.text(i, v + 0.01, f'{v:.2f}s', ha='center', fontsize=10)
plt.tight_layout()
plt.show()


---
## Phase 4: Model Evaluation & XAI

Detailed evaluation: Confusion Matrix, ROC-AUC (binary or one-vs-rest for multi-class), misclassification analysis, and feature importance. Everything below adapts to however many classes `class_names` contains.

In [ ]:
# --- Pick best model for detailed evaluation (by weighted F1) ---
best_model_name = results_df.iloc[0]['Model']
best_clf = trained_models[best_model_name]
y_pred_best = best_clf.predict(x_test)
print(f'Best model: {best_model_name}')
print(f'\n=== Detailed Classification Report ===')
report = classification_report(y_test, y_pred_best, target_names=class_names, output_dict=True, zero_division=0)
pd.DataFrame(report).transpose()


In [ ]:
# --- Confusion Matrix (generalized to N classes) ---
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5 + len(class_names), 4 + len(class_names) // 2))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test, y_pred_best, class_names, f'Confusion Matrix — {best_model_name}')


In [ ]:
# --- ROC-AUC: binary uses a single curve, multi-class uses one-vs-rest ---
if hasattr(best_clf, 'predict_proba'):
    y_prob = best_clf.predict_proba(x_test)

    if n_classes == 2:
        fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
        auc_score = roc_auc_score(y_test, y_prob[:, 1])

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f'ROC curve (AUC = {auc_score:.4f})', lw=2)
        ax.plot([0, 1], [0, 1], 'k--', label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC Curve — {best_model_name}')
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f'AUC Score: {auc_score:.4f}')
    else:
        y_test_bin = label_binarize(y_test, classes=range(n_classes))
        auc_macro = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
        auc_weighted = roc_auc_score(y_test_bin, y_prob, average='weighted', multi_class='ovr')

        fig, ax = plt.subplots(figsize=(7, 6))
        for i, cname in enumerate(class_names):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
            class_auc = roc_auc_score(y_test_bin[:, i], y_prob[:, i])
            ax.plot(fpr, tpr, label=f'{cname} (AUC={class_auc:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', label='Random')
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC Curves (One-vs-Rest) — {best_model_name}')
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f'Macro-average AUC: {auc_macro:.4f} | Weighted-average AUC: {auc_weighted:.4f}')
else:
    print(f'{best_model_name} does not support probability predictions.')


In [ ]:
# --- Confusion Matrices for ALL models ---
n_models = len(trained_models)
n_cols = 3
n_rows = -(-n_models // n_cols)  # ceil division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = np.array(axes).ravel()

for i, (name, clf) in enumerate(trained_models.items()):
    y_pred = clf.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[i], colorbar=False)
    axes[i].set_title(name)

for j in range(n_models, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# --- Misclassification Analysis (generalized to N classes, not just FP/FN on 0/1) ---
y_pred_all = best_clf.predict(x_test)
test_texts = test_df['processed_text'].values
test_labels = y_test

misclassified_idx = np.where(y_pred_all != test_labels)[0]
print(f'Total misclassified: {len(misclassified_idx)} / {len(test_labels)} '
      f'({len(misclassified_idx) / len(test_labels):.1%})')

for true_cls in range(n_classes):
    for pred_cls in range(n_classes):
        if true_cls == pred_cls:
            continue
        idxs = np.where((test_labels == true_cls) & (y_pred_all == pred_cls))[0]
        if len(idxs) == 0:
            continue
        print(f'\n=== True: {class_names[true_cls]}  ->  Predicted: {class_names[pred_cls]} '
              f'({len(idxs)} cases) ===')
        for idx in idxs[:5]:
            print(f'  [{idx}] {test_texts[idx][:200]}...')


In [ ]:
# --- Feature Importance (handles binary AND multi-class coefficient shapes) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

tree_models = [('Random Forest', trained_models.get('Random Forest')),
               ('Decision Tree', trained_models.get('Decision Tree')),
               ('AdaBoost', trained_models.get('AdaBoost')),
               ('Logistic Regression', trained_models.get('Logistic Regression'))]

ax_idx = 0
for name, clf in tree_models:
    if clf is None:
        continue
    if hasattr(clf, 'coef_'):
        # multi-class LogisticRegression has one coef row per class; average |coef| across
        # classes so a single importance ranking makes sense regardless of n_classes
        importances = np.abs(clf.coef_).mean(axis=0) if clf.coef_.shape[0] > 1 else np.abs(clf.coef_[0])
    elif hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    else:
        continue

    top_n = 15
    top_idx = np.argsort(importances)[-top_n:]

    axes[ax_idx].barh(range(top_n), importances[top_idx], color='steelblue')
    axes[ax_idx].set_yticks(range(top_n))
    axes[ax_idx].set_yticklabels([feature_names[i] for i in top_idx])
    axes[ax_idx].set_title(f'Top {top_n} Features — {name}')
    axes[ax_idx].invert_yaxis()
    ax_idx += 1

for j in range(ax_idx, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# --- Custom Input Prediction ---
def clean_single_text(text):
    if not isinstance(text, str):
        return ''
    text = contraction_expansion(text)
    text = remove_special_chars(text)
    text = remove_urls(text)
    text = remove_stopwords_from_text(text)
    return text

user_input = input('Enter a review to analyze: ')
if user_input.strip():
    cleaned = clean_single_text(user_input)
    vec = vect.transform([cleaned])
    pred = best_clf.predict(vec)[0]
    label = le.inverse_transform([pred])[0]

    print(f'\n{"="*50}')
    print(f'Review: {user_input}')
    print(f'Cleaned: {cleaned}')
    print(f'Prediction: {label}')
    if hasattr(best_clf, 'predict_proba'):
        proba = best_clf.predict_proba(vec)[0]
        for cname, p in zip(class_names, proba):
            print(f'  P({cname}) = {p:.2%}')
    print(f'{"="*50}')
else:
    print('No input entered.')


In [ ]:
# --- Compare predictions across all models ---
user_input = input('\nEnter another review (or press Enter to skip): ') if 'user_input' not in dir() else user_input
if user_input and user_input.strip():
    cleaned = clean_single_text(user_input)
    vec = vect.transform([cleaned])
    print(f'{"Model":<25} {"Prediction":<15} {"Confidence":<10}')
    print('-' * 50)
    for name, clf in trained_models.items():
        p = clf.predict(vec)[0]
        lbl = le.inverse_transform([p])[0]
        if hasattr(clf, 'predict_proba'):
            conf = clf.predict_proba(vec)[0][int(p)]
            print(f'{name:<25} {lbl:<15} {conf:.2%}')
        else:
            print(f'{name:<25} {lbl:<15} N/A')
else:
    print('Skipped.')


---
*Notebook generated from SACR Tool pipeline — all 4 phases in one end-to-end flow, fixed for neutral-class support, no train/test leakage, and generalization across binary/multi-class and balanced/imbalanced datasets.*